# Chapter 35: a constructed execution ledger

A request to buy 100 shares uses TWAP over two slots, each with forecast volume of 200 shares and maximum participation of 0.2. Each slot fills 40 shares, giving 80 filled and 20 remaining. Prices are USD per share and amounts are USD. Forecasts are frozen before the start; fills occur inside slots rather than at slot starts.

Post both fills to a ledger starting with USD 20,000 cash. Check cash against traded notional plus commission, then check NAV=cash+80×103 at the cutoff. Save constructed_ledger.json and one cash/inventory/NAV PNG under RUN_DIR. This fresh example calls the shared execution core; it does not validate Projects 4–6 or a market strategy.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.chdir(ROOT)
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch35_execution_ledger_constructed_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Default offline teaching; current empirical French execution SKIPPED (permissions unresolved).")
print("Run directory:", RUN_DIR)


In [ ]:
import datetime as dt
from dataclasses import asdict
import json
import matplotlib.pyplot as plt
from quant_course.execution_costs import ExecutionConfig, ParentOrder, ExecutionSlot, execute_order, ExecutionPortfolioLedger
start=dt.datetime(2026,1,2,14,30,tzinfo=dt.timezone.utc)
parent=ParentOrder("CONSTRUCTED-01","SIM-A",100.,start-dt.timedelta(minutes=1),start,100.,101.)
slots=[ExecutionSlot(i,start+dt.timedelta(minutes=i),start+dt.timedelta(minutes=i+1),start+dt.timedelta(minutes=i),start+dt.timedelta(minutes=i,seconds=30),200.,200.,200.,101.+i,observation_time=start-dt.timedelta(seconds=1),forecast_frozen_at=start-dt.timedelta(seconds=1)) for i in range(2)]
cfg=ExecutionConfig(commission_rate=.001,half_spread_bps=10.,temp_impact_coef=0.,perm_impact_coef=0.,max_participation=.2,cash_rate_annual=0.,borrow_rate_annual=0.)
result=execute_order(parent,"TWAP",slots,cfg,103.,start+dt.timedelta(minutes=3))
assert result.signed_filled_quantity==80. and result.signed_remainder_quantity==20.


In [ ]:
ledger=ExecutionPortfolioLedger(initial_cash=20000.,config=cfg,start_time=start)
for fill in result.fills:
    ledger.post_fill("SIM-A",fill,{"SIM-A":fill.mid_price})
state=ledger.snapshot_state()
audit=ledger.reconcile_ledger()
expected_cash=20000.-sum(f.signed_quantity*f.fill_price+f.commission for f in result.fills)
assert abs(ledger.cash-expected_cash)<1e-8
assert ledger.positions["SIM-A"]==80.
assert abs(ledger.compute_nav({"SIM-A":103.})-(expected_cash+80.*103.))<1e-8
print("CONSTRUCTED ledger",state,audit)
(RUN_DIR/"constructed_ledger.json").write_text(json.dumps({"data_kind":"CONSTRUCTED","state":state,"audit":audit},default=str,indent=2))
fig,ax=plt.subplots();ax.bar(["Cash","Inventory at cutoff","NAV"],[expected_cash,80.*103.,ledger.compute_nav({"SIM-A":103.})]);ax.set(ylabel="USD",title="CONSTRUCTED self-financing ledger");fig.tight_layout();fig.savefig(RUN_DIR/"constructed_ledger.png");plt.close(fig)
